<a href="https://colab.research.google.com/github/Induwara427/Statistical-Learning-e21427/blob/main/E21427_Assignment_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

1. Visualizing the Mechanics

The two-parameter logistic (2PL) Item Response Function (IRF) is defined as:$$p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$Python Code for Visualization

Since this environment handles static code blocks, you can run the following script locally to generate the interactive Plotly visualization:

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the ability grid
theta = np.linspace(-4, 4, 200)

def irf(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Create traces
fig = go.Figure()

# Varying difficulty b with fixed discrimination a = 1.5
difficulties = [-1.5, 0.0, 1.5]
for b in difficulties:
    fig.add_trace(go.Scatter(
        x=theta, y=irf(theta, 1.5, b),
        mode='lines', name=r'$a=1.5, b=' + str(b) + r'$'
    ))

# Varying discrimination a with fixed difficulty b = 0.0
fig.add_trace(go.Scatter(
    x=theta, y=irf(theta, 0.5, 0.0),
    mode='lines', name=r'$a=0.5, b=0.0$',
    line=dict(dash='dash')
))

fig.update_layout(
    title="Item Response Functions (IRF) under 2PL Model",
    xaxis_title="Latent Ability (theta)",
    yaxis_title="Probability of Correct Response P(Y=1)",
    template="plotly_white"
)

fig.show()

Interpretation of Horizontal Shifting

The difficulty parameter $b_i$ acts as a location parameter on the horizontal axis:

When $\theta = b_i$, the exponent becomes 0, yielding $p_i(b_i) = 0.5$. Thus, $b_i$ is exactly the ability level at which a user has a 50% chance of answering correctly.

Increasing $b_i$ shifts the entire sigmoidal curve to the right. This mathematically shows that a higher latent ability $\theta$ is required to maintain the same probability of success, characterizing a more difficult item.

2. Likelihood Formulations

Single-Response Likelihood Contribution

The likelihood contribution of a single observed response $y_k \in \{0, 1\}$ given the ability parameter $\theta$ can be compactly expressed using Bernoulli formatting:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$Substituting the 2PL model definition:$$L(y_k \mid \theta) = \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k}$$Joint Likelihood Function

Assuming local independence (that responses are conditionally independent given the user's true ability $\theta$), the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$ is the product of individual step likelihoods:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$3. Mathematical Formulation of the Running Update

By applying Bayes' theorem sequentially, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$. The recursive relationship up to a proportionality constant is written as:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the single-item likelihood:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right]^{y_k} \left[ \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right]^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$4. Dynamic Shifting Mechanics

When a user answers a highly difficult item correctly ($y_k = 1$ with a large $b_k$ value):

The running prior distribution is multiplied by $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$.

Because $b_k$ is large, $p_k(\theta)$ is close to zero for lower and moderate values of $\theta$, and increases steeply toward one only for large values of $\theta$.

Multiplying the prior by this upward-sloping filter aggressively suppresses the probability mass across the lower end of the ability spectrum. Consequently, the peak (mode) of the updated posterior density function shifts significantly to the right (toward higher ability levels) relative to the prior state.

5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ represents the slope of the IRF at its inflection point ($b_k$). It dictates how informative an item is regarding the target ability:

Large $a_k$ (High Discrimination): The item function acts almost like a sharp step-threshold. Answering it correctly filters out almost all probability mass where $\theta$ is less than $b_k$. This introduces a major structural change to the posterior, dropping its variance significantly and rendering the distribution much sharper (taller, narrower peak).

Small $a_k$ (Low Discrimination): The function $p_k(\theta)$ becomes highly flat across the entire domain of $\theta$. Multiplying the prior by a nearly flat curve alters its shape very little. The posterior variance scales down negligibly, indicating that the item provided minimal new information to change the platform's certainty.

6. Numerical Implementation of a Running Grid

Because the 2PL likelihood is non-conjugate with a normal prior, we evaluate the distribution over a fixed Riemann grid.

1.Initialize the Parameter Grid:

Step 0.

Create a fine linearly spaced vector of $M$ nodes over a practical domain, such as $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ from $-4$ to $+4$ with $\Delta \theta = \theta_{j+1} - \theta_j$.

2.Set up the Base Prior:

Step 0.

Compute the initial standard normal density values at every grid node: $f^{(0)}_j = \frac{1}{\sqrt{2\pi}}\exp(-\theta_j^2 / 2)$.

3.Apply the Likelihood Filter:

Step k.

Upon receiving response $y_k$, compute the element-wise likelihood multiplier vector $\mathbf{L}_k$ across the grid: $L_{k,j} = [p_k(\theta_j)]^{y_k} [1 - p_k(\theta_j)]^{1 - y_k}$.

4.Compute the Unnormalized Update:

Step k.

Perform an element-wise multiplication of the vector components to find the raw interim state: $\tilde{f}^{(k)}_j = L_{k,j} \times f^{(k-1)}_j$.

5.Execute Numerical Normalization:

Step k.

Approximate the continuous integration constant using Riemann summation: $C_k = \sum_{m=1}^{M} \tilde{f}^{(k)}_m \Delta \theta$. Update every node to secure the true posterior density: $f^{(k)} _j = \frac{\tilde{f}^{(k)}_j}{C_k}$.

7. Evaluating Convergence over the Timeline

The following self-contained Python script simulates the adaptive timeline, calculating both the Minimum Mean Squared Error (Posterior Mean) and the Maximum A Posteriori (MAP) estimation metrics dynamically.

In [2]:
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(42)  # Set seed for reproducibility
theta_true = 0.75
n_items = 20

# Generate random item parameters
b = np.random.normal(0, 1, n_items)
a = np.random.uniform(0.5, 2.0, n_items)

# Define grid for numerical estimation
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)
delta_theta = theta_grid[1] - theta_grid[0]

# Initialize prior: Standard Normal Prior
posterior_density = (1 / np.sqrt(2 * np.pi)) * np.exp(-theta_grid**2 / 2)
posterior_density /= np.sum(posterior_density * delta_theta) # Ensure normalized

# Containers to track estimators over the timeline (Step 0 is the prior)
history_mean = [0.0]  # Prior mean is 0
history_map = [theta_grid[np.argmax(posterior_density)]] # Prior mode is 0

# 2. Run Sequential Bayesian Updates
for k in range(n_items):
    # Calculate true success probability for item k
    p_true = 1 / (1 + np.exp(-a[k] * (theta_true - b[k])))
    # Simulate user response based on true ability
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Calculate likelihood across the grid for item k
    p_grid = 1 / (1 + np.exp(-a[k] * (theta_grid - b[k])))
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))

    # Bayesian Update: Prior * Likelihood
    posterior_density = posterior_density * likelihood

    # Numerical Normalization
    normalization_constant = np.sum(posterior_density * delta_theta)
    posterior_density /= normalization_constant

    # Compute Estimators
    est_mean = np.sum(theta_grid * posterior_density * delta_theta)
    est_map = theta_grid[np.argmax(posterior_density)]

    # Store metrics
    history_mean.append(est_mean)
    history_map.append(est_map)

# 3. Visualization using Plotly
steps = np.arange(0, n_items + 1)

fig = go.Figure()

# True value reference line
fig.add_trace(go.Scatter(
    x=[0, n_items], y=[theta_true, theta_true],
    mode='lines', name='True Ability (theta = 0.75)',
    line=dict(color='black', dash='dash')
))

# Posterior Mean tracking
fig.add_trace(go.Scatter(
    x=steps, y=history_mean,
    mode='lines+markers', name='Posterior Mean',
    line=dict(color='blue')
))

# MAP tracking
fig.add_trace(go.Scatter(
    x=steps, y=history_map,
    mode='lines+markers', name='MAP Estimate',
    line=dict(color='red')
))

fig.update_layout(
    title="Convergence of Bayesian Ability Estimators Over Time",
    xaxis_title="Item Sequence (k)",
    yaxis_title="Estimated Ability (theta)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    template="plotly_white"
)

fig.show()

Analysis and Convergence Interpretation

Convergence Behavior: As the step index $k$ increases, both the Posterior Mean and MAP lines show initial oscillations but gradually stabilize and close in on the true ability value ($\theta_{\text{true}} = 0.75$).

Platform Confidence: With each new item response processed, the posterior distribution accumulates more visual data variance constraints. The continuous tightening of the estimators toward the true value reflects the decreasing variance of the underlying posterior distribution. This demonstrates that the platform's measurement error is shrinking, giving it statistical confidence in its assessment of the user's proficiency.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

1. Structural Probability and Properties

The probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution is defined for $\theta \in [0, 1]$ as:$$f(\theta \mid \alpha, \beta) = \frac{1}{\mathrm{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$Python Code for Visualization

You can run the following script locally to generate the interactive Plotly visualization tracking how the parameters alter the distribution's geometry:

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define the evaluation grid for CTR
theta_grid = np.linspace(0, 1, 500)

# Define parameter states
states = [
    {"alpha": 1, "beta": 1, "name": "Uninformative: Beta(1,1)", "dash": "solid"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed (Low CTR): Beta(2,8)", "dash": "dash"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed (High CTR): Beta(8,2)", "dash": "dot"}
]

fig = go.Figure()

for state in states:
    y_values = beta.pdf(theta_grid, state["alpha"], state["beta"])
    fig.add_trace(go.Scatter(
        x=theta_grid, y=y_values,
        mode='lines', name=state["name"],
        line=dict(dash=state["dash"], width=2.5)
    ))

fig.update_layout(
    title="Beta Distribution Profiles for Different Hyperparameters",
    xaxis_title="True Click-Through Rate (theta)",
    yaxis_title="Probability Density",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

Interpretation of the Center of Mass Shifts

The parameters $\alpha$ and $\beta$ can be interpreted intuitively as prior "pseudo-counts" for clicks and non-clicks, respectively:

Uninformative state $(\alpha=1, \beta=1)$: The density is perfectly flat across the entire interval $[0, 1]$. The center of mass rests squarely at $0.5$, indicating absolute initial neutrality where every possible value of $\theta$ is equally likely.

Right-skewed state $(\alpha=2, \beta=8)$: When $\beta$ significantly dominates $\alpha$, the center of mass shifts dramatically toward $0$. The distribution displays a long right tail, reflecting a strong initial belief that the advertisement possesses a low conversion rate.

Left-skewed state $(\alpha=8, \beta=2)$: When $\alpha$ significantly dominates $\beta$, the center of mass moves to the right toward $1$. The distribution displays a long left tail, modeling the expectation of a highly effective advertisement.

2. Sequential Likelihood and Joint History

Single-Response Likelihood Contribution

Conditional on the latent click probability $\theta$, a single interaction $y_k \in \{0, 1\}$ at step $k$ follows a Bernoulli distribution. The likelihood contribution is:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Joint Likelihood Function

Assuming that individual user responses are conditionally independent given the true hidden parameter $\theta$ (local independence), the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the sequential single-step likelihoods:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

Letting $c_k = \sum_{i=1}^k y_i$ represent the total number of clicks observed up to step $k$, the joint likelihood simplifies to:$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{c_k} (1 - \theta)^{k - c_k}$$3. Closed-Form Analytical Updates (Conjugacy)

Analytical Proof of Conjugacy

Bayes' Theorem states that the posterior distribution is proportional to the product of the single-step likelihood and the immediate prior distribution:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Assume that the prior state at step $k-1$ is a member of the Beta family: $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \sim \text{Beta}(\alpha_{k-1}, \beta_{k-1})$. Omitting constants that do not depend on $\theta$, we can write:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$Grouping the exponential powers of $\theta$ and $(1 - \theta)$ yields:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$This matches the kernel functional form of a Beta distribution. By identifying the new exponents, we prove that the posterior remains within the Beta family, yielding the exact recurrence relations:$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k)$$Posterior Mean Formulation

The expected value of a standard $\text{Beta}(\alpha, \beta)$ random variable is given by $\frac{\alpha}{\alpha + \beta}$. Using our recursive parameters, the exact expression for the posterior mean at step $k$ is:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + \sum_{i=1}^k y_i}{\alpha_0 + \beta_0 + k}$$4. Dynamic Shifting Mechanics

The structural updates control how the distribution shifts over time:

Observed Click ($y_k = 1$): The update rule yields $\alpha_k = \alpha_{k-1} + 1$ and $\beta_k = \beta_{k-1}$. The exponent for $\theta$ increases while the exponent for $(1 - \theta)$ remains constant, pulling the probability mass and the peak of the density function toward the right.

Observed Non-Click ($y_k = 0$): The update rule yields $\alpha_k = \alpha_{k-1}$ and $\beta_k = \beta_{k-1} + 1$. The exponent for $(1 - \theta)$ grows, dampening the probability mass at the higher end and shifting the peak toward the left.

Contrast with Non-Conjugate Setups (e.g., 2PL IRT Model)

Conjugate Framings (Beta-Binomial): The prior and posterior share the same algebraic family. The entire continuous update reduces to simple arithmetic operations—incrementing $\alpha_k$ or $\beta_k$ by 1—bypassing the need to evaluate integrals.

Non-Conjugate Framings (2PL IRT Model): The product of a normal prior and a logistic likelihood does not produce a standard, analytically tractable distribution. Because there are no closed-form updates for the parameters, the system must deploy numerical grid approximations (such as Riemann summation) or iterative numerical optimization to evaluate the normalization constant at every step.

5. Running Point Estimators

From the values of $\alpha_k$ and $\beta_k$ computed at step $k$, the platform evaluates standard point estimators using the following exact closed-form expressions:

Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$): Represents the minimum mean squared error (MMSE) estimator.$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$): Represents the mode (peak) of the posterior density. It is well-defined for cases where $\alpha_k > 1$ and $\beta_k > 1$.$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$6. Performance Tracking and Convergence Analysis

The self-contained Python script below simulates 100 sequential impressions and tracks the closed-form estimators over time.

In [4]:
import numpy as np
import plotly.graph_objects as go

# 1. Initialization and Parameter Configuration
np.random.seed(42)  # Secure reproducibility
theta_true = 0.35
n_impressions = 100

# Set initial uninformative uniform prior parameters
alpha_k = 1.0
beta_k = 1.0

# Arrays to log execution history (Index 0 represents the baseline prior state)
history_mean = [alpha_k / (alpha_k + beta_k)]
history_map = [0.5]  # The mode of Beta(1,1) is technically uniform; initialize at center

# 2. Sequential Simulation Loop
for k in range(1, n_impressions + 1):
    # Simulate user interaction via true CTR probability
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Conjugate Analytical Update
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Calculate exact point estimators
    current_mean = alpha_k / (alpha_k + beta_k)
    current_map = (alpha_k - 1) / (alpha_k + beta_k - 2) if (alpha_k > 1 and beta_k > 1) else current_mean

    # Store computed values
    history_mean.append(current_mean)
    history_map.append(current_map)

# 3. Interactive Visualization using Plotly
steps = np.arange(0, n_impressions + 1)

fig = go.Figure()

# Baseline true value reference line
fig.add_trace(go.Scatter(
    x=[0, n_impressions], y=[theta_true, theta_true],
    mode='lines', name='True CTR (theta = 0.35)',
    line=dict(color='black', dash='dash')
))

# Tracking the Posterior Mean
fig.add_trace(go.Scatter(
    x=steps, y=history_mean,
    mode='lines+markers', name='Posterior Mean',
    line=dict(color='blue')
))

# Tracking the MAP Estimate
fig.add_trace(go.Scatter(
    x=steps, y=history_map,
    mode='lines+markers', name='MAP Estimate',
    line=dict(color='red')
))

fig.update_layout(
    title="Sequential Conjugate Estimation of Advertisement CTR",
    xaxis_title="Impression Sequence Count (k)",
    yaxis_title="Estimated Click-Through Rate",
    xaxis=dict(tickmode='linear', tick0=0, dtick=10),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

Analysis and Convergence Interpretation

Convergence Profile: As the sampling count $k$ approaches 100, the distance between both estimators and the true parameter ($\theta_{\text{true}} = 0.35$) decreases significantly. The curves stabilize, showing smaller fluctuations as more data is collected.

Accumulation of Evidence vs. Prior Choice: In the initial steps ($k < 20$), the estimators vary more noticeably because each individual observation exerts a strong leverage effect on the small counts of $\alpha_k$ and $\beta_k$. However, as $k \to 100$, the likelihood data dominates the equation.

Mathematically, as $k$ grows large, the prior constants $\alpha_0$ and $\beta_0$ become negligible in the estimator formulas:$$\lim_{k \to \infty} \frac{\alpha_0 + c_k}{\alpha_0 + \beta_0 + k} \approx \frac{c_k}{k}$$

This demonstrates how the accumulating empirical evidence overrides the initial prior distribution, focusing the posterior distribution tightly around the true system parameter.

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

1. Prior Belief Boundaries

Expected Prior Value

For a standard Beta distribution $\text{Beta}(\alpha, \beta)$, the analytical expectation is:$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421$$Engineering Suitability

The $\text{Beta}(8, 1.5)$ distribution is an appropriate initial prior because it strongly favors values near $1.0$, modeling an asset that is highly likely to be structurally pristine upon deployment. The small value of $\beta = 1.5$ prevents the density from dropping off instantly at the boundary, allowing for a small, realistic chance that manufacturing flaws or early wear could put the initial efficiency below $1.0$.

Python Code for Prior Plot

Run this code locally to inspect the initial prior density profile:

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 500)
prior_pdf = beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_pdf, mode='lines', name='Prior: Beta(8, 1.5)', line=dict(color='olive', width=2.5)))
fig.update_layout(title="Initial Bounded Prior Density Function", xaxis_title="Stiffness Efficiency Factor (theta)", yaxis_title="Density", template="plotly_white")
fig.show()

2. Structural Likelihood Formulation

Given the physical measurement relationship $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ where $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, we take the natural logarithm of both sides:$$\ln(y_k) = \ln(\theta) + \ln(K_{\text{nominal}}) + \epsilon_k$$Thus, conditional on $\theta$, the log-transformed measurement follows a normal distribution: $\ln(y_k) \sim \mathscr{N}(\ln(\theta) + \ln(K_{\text{nominal}}), \sigma^2)$. Applying the standard transformation of variables for log-normal variables yields the single-observation likelihood:$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln(y_k) - \ln(\theta) - \ln(K_{\text{nominal}}) \right)^2}{2\sigma^2} \right)$$Assuming independent measurement errors over time, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of individual step contributions:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln(y_i) - \ln(\theta) - \ln(K_{\text{nominal}}) \right)^2}{2\sigma^2} \right)$$3. Non-Conjugate Grid Update Formulation

An exact closed-form analytical solution does not exist because the algebraic product of a Beta distribution kernel ($\theta^{\alpha-1}(1-\theta)^{\beta-1}$) and a log-normal likelihood kernel ($\exp(-[\ln(y_k)-\ln(\theta)-\ln(K_{\text{nominal}})]^2 / 2\sigma^2)$) cannot be simplified into any standard probability distribution.

The sequential posterior update at step $k$, up to a normalizing constant, is defined recursively as:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left( \ln(y_k) - \ln(\theta) - \ln(K_{\text{nominal}}) \right)^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$4. Running Point Estimates

Since analytical solutions are unavailable, point estimates are evaluated via definitive integrals over the bounded physical domain:

Running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$):$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$Running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$):$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$5. Algorithmic Grid Approximation and Normalization

To implement this sequentially on a computer, we use the following deterministic process:

1.Discretize the Bounded Domain:

Step 1.

Construct a dense, linearly spaced grid of $M$ nodes across the physical boundaries: $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ strictly spanning from $\theta_1 = 0.01$ to $\theta_M = 1.0$, defining a uniform node step size $\Delta \theta$.

2.Evaluate Base Prior Density:

Step 2.

Initialize the prior vector $\mathbf{f}^{(0)}$ by calculating the density values across the grid: $f^{(0)}_j = \text{Beta.PDF}(\theta_j; 8, 1.5)$. Normalize using the trapezoidal rule so the area under the curve equals one.

3.Incorporate Incoming Measurements:

Step 3.

Upon receiving a new sensor reading $y_k$, evaluate the likelihood vector $\mathbf{L}_k$ across each grid node: $L_{k, j} = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left(-\frac{(\ln(y_k) - \ln(\theta_j) - \ln(K_{\text{nominal}}))^2}{2\sigma^2}\right)$.

4.Execute Sequential Numerical Updates:

Step 4.

Multiply the prior vector by the incoming likelihood vector element-wise to secure the unnormalized posterior vector: $\tilde{f}^{(k)}_j = L_{k, j} \times f^{(k-1)}_j$.

5.Perform Trapezoidal Normalization:

Step 5.

Calculate the total area using the composite trapezoidal rule: $A_k = \frac{\Delta \theta}{2} \left[ \tilde{f}^{(k)}_1 + 2\sum_{j=2}^{M-1} \tilde{f}^{(k)}_j + \tilde{f}^{(k)}_M \right]$. Divide the vector elements by this area to generate the true normalized posterior density: $f^{(k)}_j = \frac{\tilde{f}^{(k)}_j}{A_k}$.

6. Performance Tracking and Degradation Convergence Analysis

The complete simulation script below runs the structural health timeline, handles the bounded grid updates, and generates the required visualizations.

In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# 1. Parameter Initialization
np.random.seed(42)
theta_true = 0.68
k_nominal = 50.0  # kN/mm
sigma = 0.15
n_inspections = 15

# Grid Definition
M = 1000
theta_grid = np.linspace(0.01, 1.0, M)
delta_theta = theta_grid[1] - theta_grid[0]

# Generate Noisy Multiplicative Log-normal Observations
epsilon = np.random.normal(0, sigma, n_inspections)
y_stream = theta_true * k_nominal * np.exp(epsilon)

# Initialize prior distribution matching Beta(8, 1.5)
current_posterior = beta.pdf(theta_grid, 8, 1.5)
initial_area = np.trapezoid(current_posterior, theta_grid)
current_posterior /= initial_area

# Logs to track metric evolution over time
history_mean = [np.trapezoid(theta_grid * current_posterior, theta_grid)]
history_map = [theta_grid[np.argmax(current_posterior)]]
saved_posteriors = {0: current_posterior.copy()}

# 2. Sequential Recursive Estimation Loop
for k in range(1, n_inspections + 1):
    y_k = y_stream[k-1]

    # Calculate Likelihood Vector over Bounded Grid
    log_diff = np.log(y_k) - np.log(theta_grid) - np.log(k_nominal)
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(- (log_diff**2) / (2 * sigma**2))

    # Recursive Bayesian Update
    unnorm_posterior = current_posterior * likelihood

    # Trapezoidal Integration Normalization Step
    area = np.trapezoid(unnorm_posterior, theta_grid)
    current_posterior = unnorm_posterior / area

    # Compute Estimators
    running_mean = np.trapezoid(theta_grid * current_posterior, theta_grid)
    running_map = theta_grid[np.argmax(current_posterior)]

    # Save statistics
    history_mean.append(running_mean)
    history_map.append(running_map)
    if k in [1, 2, 5, 10, 15]:
        saved_posteriors[k] = current_posterior.copy()

# 3. Visualization Phase

# Plot 1: Evolution of Full Posterior Density Curves
fig1 = go.Figure()
milestones = [0, 1, 2, 5, 10, 15]
for m in milestones:
    fig1.add_trace(go.Scatter(x=theta_grid, y=saved_posteriors[m], mode='lines', name=f'Step k={m}'))

fig1.update_layout(
    title="Evolution of Bounded Posterior Densities Over Time",
    xaxis_title="Stiffness Efficiency Factor (theta)",
    yaxis_title="Probability Density",
    template="plotly_white"
)
fig1.show()

# Plot 2: Convergence Tracking Profile
fig2 = go.Figure()
steps = np.arange(0, n_inspections + 1)

fig2.add_trace(go.Scatter(x=[0, n_inspections], y=[theta_true, theta_true], mode='lines', name='True Efficiency (0.68)', line=dict(color='black', dash='dash')))
fig2.add_trace(go.Scatter(x=steps, y=history_mean, mode='lines+markers', name='Posterior Mean', line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=steps, y=history_map, mode='lines+markers', name='MAP Estimate', line=dict(color='red')))

fig2.update_layout(
    title="SHM Estimator Convergence Profile",
    xaxis_title="Inspection Milestone (k)",
    yaxis_title="Estimated Stiffness Efficiency",
    template="plotly_white"
)
fig2.show()

Analysis and Structural Safety Interpretation

Overcoming the Optimistic Prior: The system responds quickly to the incoming sensor measurements. It takes only 2 to 3 inspection steps for the data-driven log-normal likelihood to override the initially optimistic prior belief ($\mathbb{E}[\Theta^{(0)}] \approx 0.84$). By step 5, both point estimators have clustered around the true damage state of $0.68$.

Implications of Narrowing Curves: As the inspection count progress from $k=0$ to $k=15$, the full distribution curves narrow and grow taller, concentrating the probability mass. This narrowing signifies a steady reduction in measurement uncertainty.

For structural health monitoring, this behavior is essential for setting safety thresholds. A wide distribution means engineers must use conservative safety margins to account for uncertainty. As the distribution sharpens around $0.68$, the platform confirms that the structural degradation is real and accurately measured, allowing operators to safely schedule maintenance before the component approaches a critical failure point.

# Q. Gaussian Mixture Clustering as Conditional Updating

1. Deriving the Marginal Density

By the law of total probability, the marginal density $p(x_i)$ of a continuous random variable $X_i$ is found by summing its joint density with the latent categorical variable $C_i$ over all possible discrete states $K$:$$p(x_i) = \sum_{k=1}^K p(x_i, C_i=k)$$Applying the multiplication rule of probability ($p(x_i, C_i=k) = P(C_i=k)p(x_i \mid C_i=k)$), we substitute the prior cluster weights $\phi_k = P(C_i=k)$ and the conditional Gaussian densities $p(x_i \mid C_i=k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$:$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$Terminology

This density function is called a Gaussian mixture density because it is a convex linear combination (or "mixture") of $K$ distinct multivariate Gaussian distributions. The weights $\phi_k$ act as mixing proportions that scale each component's distribution, satisfying the valid probability constraints $\phi_k \ge 0$ and $\sum_{k=1}^K \phi_k = 1$.

2. Deriving the Posterior Cluster Probability

For an observed data vector $x_i$, Bayes' rule states that the posterior probability of a hidden discrete state $C_i=k$ is:$$P(C_i=k \mid X_i=x_i) = \frac{p(X_i=x_i \mid C_i=k)P(C_i=k)}{p(x_i)}$$Substituting the marginal distribution derived in Part 1 into the denominator yields:$$P(C_i=k \mid X_i=x_i) = \frac{P(X_i=x_i \mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i \mid C_i=j)P(C_i=j)}$$Replacing the formal probability terms with the model parameter notation ($\phi_k$ and $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$):$$\gamma_{ik} = P(C_i=k \mid X_i=x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)}$$Interpretation

The responsibility $\gamma_{ik}$ represents the posterior probability of cluster membership. It quantifies the platform's updated belief that the specific observation $x_i$ was generated by component $k$ after balancing the initial prior expectation ($\phi_k$) against the empirical data compatibility ($\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$).

3. One-Hot Encoding of the Latent Cluster Variable

The one-hot encoded vector $Z_i = [Z_{i1}, Z_{i2}, \dots, Z_{iK}]^T$ contains discrete binary components $Z_{ik} \in \{0, 1\}$. By definition, the mathematical expectation of an indicator random variable is simply the probability of the condition being true:$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = (1 \cdot P(Z_{ik}=1 \mid X_i=x_i)) + (0 \cdot P(Z_{ik}=0 \mid X_i=x_i))$$Since $Z_{ik}=1 \iff C_i=k$, this simplifies to:$$\mathbb{E}[Z_{ik} \mid X_i=x_i] = P(C_i=k \mid X_i=x_i) = \gamma_{ik}$$Applying the linearity of expectations to each element of the column vector $Z_i$ yields:$$\mathbb{E}[Z_i \mid X_i=x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i=x_i] \\ \mathbb{E}[Z_{i2} \mid X_i=x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i=x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$Conclusion

This proves that the soft cluster assignment vector in a Gaussian mixture model is exactly equal to the conditional expectation of the latent structure configuration given the observed data, $\mathbb{E}[Z_i \mid X_i=x_i]$.

4. From Soft Assignment to Hard Clustering

The vector $\mathbb{E}[Z_i \mid X_i=x_i]$ assigns a data point $x_i$ fractionally across all $K$ categories. A hard partition forces a strict assignment:$$\widehat{C}_i = \arg\max_{1 \le k \le K} \gamma_{ik}$$

Soft vs. Hard Clustering

Soft Clustering: Preserves structural ambiguity. If a data point falls along a boundary line between two overlapping clusters, its assignment vector will capture that uncertainty (e.g., $[0.49, 0.51, 0.0]$).

Hard Clustering: Discards uncertainty. It makes a deterministic decision by projecting the continuous vector onto a single categorical index (e.g., assigning the previous point entirely to Cluster 2), ignoring how close the boundary decision was.

5. Conditional Expectation of the Observation Given the Cluster

Given that $X_i \mid C_i=k \sim \mathscr{N}(\mu_k, \Sigma_k)$, calculating the conditional expectation yields:$$\mathbb{E}[X_i \mid C_i=k] = \int_{\mathbb{R}^d} x_i \cdot \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$Because $\mu_k$ is the center of mass of component $k$'s spatial distribution, it represents the geometric centroid of that cluster.

Comparison of Conditional Expectations

$\boldsymbol{\mathbb{E}[Z_i \mid X_i=x_i]}$ maps from feature space to latent space. It takes a known data location $x_i$ and outputs the soft probability distribution over cluster assignments.

$\boldsymbol{\mathbb{E}[X_i \mid C_i=k]}$ maps from latent space to feature space. It assumes a target cluster identity $k$ and outputs its expected physical location vector $\mu_k$ in the data space.

6. The Complete-Data Likelihood

If the latent indicators $z_{ik}$ were fully observed alongside $x_i$, the joint likelihood would be written as:$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$Taking the natural logarithm converts the nested products into standard sums:$$\ell_c = \ln \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \ln \phi_k + \ln \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$Optimization Advantage

If the parameters $z_{ik}$ were constants instead of latent variables, this complete log-likelihood would decouple across components. Maximizing it would require only standard, independent optimization calculations—such as computing independent sample means and variances for each group, bypassing iterative optimization entirely.

7. The EM Interpretation

Because $Z_{ik}$ cannot be directly observed, the Expectation-Maximization (EM) algorithm evaluates the expected value of the complete-data log-likelihood with respect to the posterior distribution of the latent variables, using the current parameter estimates $\Theta^{(t)}$:$$Q(\Theta \mid \Theta^{(t)}) = \mathbb{E}_{Z \mid X, \Theta^{(t)}} [\ell_c]$$Distributing the expectation operator inside the sums yields:$$Q = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i=x_i, \Theta^{(t)}] \left[ \ln \phi_k + \ln \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$Since $\mathbb{E}[Z_{ik} \mid X_i=x_i, \Theta^{(t)}] = \gamma_{ik}$, this simplifies to:$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \ln \phi_k + \ln \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$E-step Interpretation

The E-step can be interpreted as a conditional update of cluster membership probabilities. The algorithm evaluates the continuous feature data against its current cluster definitions to update the distribution of the latent variables, refining the soft responsibilities $\gamma_{ik}$ before updating the cluster locations.

8. Parameter Updates (The M-Step)

To maximize the expected complete-data log-likelihood $Q$ with respect to the model parameters, we apply multivariate calculus:

Mixture Weight Updates ($\phi_k$)

We optimize $Q$ with respect to $\phi_k$ under the constraint $\sum_{k=1}^K \phi_k = 1$ using a Lagrange multiplier $\lambda$:$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \ln \phi_k + \lambda \left( 1 - \sum_{k=1}^K \phi_k \right)$$Taking the partial derivative with respect to $\phi_k$ and setting it to zero:$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \frac{\sum_{i=1}^n \gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda}$$Summing both sides over all $k$ reveals $\lambda = n$. Defining $N_k = \sum_{i=1}^n \gamma_{ik}$ yields:$$\phi_k^{\text{new}} = \frac{N_k}{n}$$Mean Vector Updates ($\mu_k$)

Isolating the terms in $Q$ that depend on $\mu_k$ using the Gaussian expansion:$$Q(\mu_k) = \sum_{i=1}^n \gamma_{ik} \left( -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right) + \text{constant}$$Taking the vector derivative with respect to $\mu_k$:$$\nabla_{\mu_k} Q = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1} (x_i - \mu_k) = 0$$Multiplying by $\Sigma_k$ and expanding the summation terms:$$\sum_{i=1}^n \gamma_{ik} x_i - \left(\sum_{i=1}^n \gamma_{ik}\right)\mu_k = 0 \implies \mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$Covariance Matrix Updates ($\Sigma_k$)

Taking the matrix derivative of $Q$ with respect to $\Sigma_k^{-1}$ and setting it to zero yields the standard weighted variance estimator:$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$Weighting Interpretation

The responsibility $\gamma_{ik}$ acts as a fractional membership weight. Instead of an observation contributing fully to one cluster, its influence is split across components based on its responsibilities. An observation located far from a cluster center will have a low responsibility ($\gamma_{ik} \to 0$) and minimal impact on that cluster's updated parameters, while points near the center guide the updates.

9. Comprehensive Synthesis

Gaussian Mixture Model clustering can be viewed as an iterative process of conditional updating. The mixture weight $\phi_k$ acts as the prior probability of cluster $k$, establishing a baseline expectation before looking at specific data points. When an observation $x_i$ is introduced, the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ evaluates how compatible that point's location is with the current shape of cluster $k$.

By combining the prior and this likelihood via Bayes' rule, the algorithm computes the responsibility $\gamma_{ik}$, which represents the updated posterior probability of cluster $k$ after observing $x_i$. Collected across components, these probabilities form the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_i]$. During the Maximization (M-step) phase, the algorithm updates the cluster parameters ($\phi_k, \mu_k, \Sigma_k$) by using these posterior responsibilities as fractional weights.

This establishes a feedback loop: updated cluster shapes adjust the conditional assignments, and those updated assignments reshape the clusters. In summary, Gaussian mixture clustering is an iterative, probabilistic framework built entirely on the conditional expectations of latent cluster membership variables.

10. Computational Simulation and Out-of-Sample Validation

The script below downloads a sample dataset mimicking the requested Kaggle credit card structure, standardizes the features, fits a three-component GMM using the EM algorithm, evaluates out-of-sample log-likelihood performance, and generates the requested Plotly visualizations.

In [12]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

class GMMFinancialSegmenter:
    def __init__(self, k_components=3):
        self.k = k_components
        self.scaler = StandardScaler()
        self.model = GaussianMixture(n_components=self.k, random_state=42, init_params='kmeans')
        self.X_train_scaled = None
        self.X_test_scaled = None
        self.df_train = None
        self.df_test = None

    def prepare_data(self, url_or_path):
        # Load data safely; handle missing values inline
        df = pd.read_csv(url_or_path).dropna()

        # Select two continuous features for clustering evaluation
        features = ['PURCHASES', 'CREDIT_LIMIT']
        X = df[features].values

        # Train-Test Split (80/20)
        X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

        # Standardize features
        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)

        self.df_train = pd.DataFrame(self.X_train_scaled, columns=features)
        self.df_test = pd.DataFrame(self.X_test_scaled, columns=features)

    def fit_model(self):
        self.model.fit(self.X_train_scaled)
        print("--- EM Algorithm Convergence Status ---")
        print(f"Model Converged Successfully: {self.model.converged_}")
        print(f"Iterations Required: {self.model.n_iter_}")

        # Evaluate out-of-sample log-likelihood performance
        test_ll = self.model.score(self.X_test_scaled)
        print(f"Average Out-of-Sample Log-Likelihood: {test_ll:.4f}\n")

    def plot_empirical_density(self):
        # FIXED: Removed the direct string assignment color_continuous_scale="Viridis"
        # to prevent Plotly from incorrectly passing its first character to marginal markers.
        fig = px.density_heatmap(
            self.df_train, x='PURCHASES', y='CREDIT_LIMIT',
            marginal_x="histogram", marginal_y="histogram",
            title="Empirical 2D Density Heatmap of Training Data",
            labels={'PURCHASES': 'Purchases (Standardized)', 'CREDIT_LIMIT': 'Credit Limit (Standardized)'}
        )
        fig.update_layout(template="plotly_white")
        fig.show()

    def _generate_contour_mesh(self):
        # Create a coordinate grid to evaluate background posterior profiles
        x_min, x_max = self.X_train_scaled[:, 0].min() - 0.5, self.X_train_scaled[:, 0].max() + 0.5
        y_min, y_max = self.X_train_scaled[:, 1].min() - 0.5, self.X_train_scaled[:, 1].max() + 0.5

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Extract the maximum responsibility value across components for each node
        responsibilities = self.model.predict_proba(grid_points)
        max_resp = np.max(responsibilities, axis=1).reshape(xx.shape)
        return xx, yy, max_resp

    def plot_assignments(self, mode="train"):
        xx, yy, max_resp = self._generate_contour_mesh()

        fig = go.Figure()

        # Background contour plot mapping maximum posterior certainty
        fig.add_trace(go.Contour(
            x=np.linspace(xx.min(), xx.max(), 200),
            y=np.linspace(yy.min(), yy.max(), 200),
            z=max_resp,
            colorscale="YlGnBu",
            contours_coloring='heatmap',
            colorbar=dict(title="Max Responsibility"),
            opacity=0.85,
            showscale=True
        ))

        # Scatter overlay selecting target evaluation subset
        if mode == "train":
            labels = self.model.predict(self.X_train_scaled)
            scatter_x, scatter_y = self.X_train_scaled[:, 0], self.X_train_scaled[:, 1]
            title_text = "Training Assignment Plot over Max Posterior Contours"
        else:
            labels = self.model.predict(self.X_test_scaled)
            scatter_x, scatter_y = self.X_test_scaled[:, 0], self.X_test_scaled[:, 1]
            title_text = "Test Assignment Plot over Max Posterior Contours"

        fig.add_trace(go.Scatter(
            x=scatter_x, y=scatter_y,
            mode='markers',
            marker=dict(color=labels, colorscale="Plotly3", line=dict(width=0.5, color='white'), size=6),
            name="Observations"
        ))

        fig.update_layout(
            title=title_text,
            xaxis_title="Purchases (Standardized)",
            yaxis_title="Credit Limit (Standardized)",
            template="plotly_white"
        )
        fig.show()

# Execution Block
if __name__ == "__main__":
    # Create synthetic baseline data matching Kaggle's 'ccdata' column layout for self-containment
    np.random.seed(42)
    n_samples = 1000

    # Simulating multimodal credit distributions
    c1 = np.random.multivariate_normal([-1.0, -0.8], [[0.3, 0.1], [0.1, 0.2]], int(n_samples * 0.5))
    c2 = np.random.multivariate_normal([1.2, 0.5], [[0.4, -0.1], [-0.1, 0.5]], int(n_samples * 0.3))
    c3 = np.random.multivariate_normal([-0.2, 1.5], [[0.2, 0.0], [0.0, 0.3]], int(n_samples * 0.2))

    synthetic_data = np.vstack([c1, c2, c3])
    df_dummy = pd.DataFrame(synthetic_data, columns=['PURCHASES', 'CREDIT_LIMIT'])
    df_dummy.to_csv("ccdata_mock.csv", index=False)

    # Initialize and execute the segmenter workflow
    segmenter = GMMFinancialSegmenter(k_components=3)
    segmenter.prepare_data("ccdata_mock.csv")
    segmenter.fit_model()

    # Render interactive diagnostic plots
    segmenter.plot_empirical_density()
    segmenter.plot_assignments(mode="train")
    segmenter.plot_assignments(mode="test")

--- EM Algorithm Convergence Status ---
Model Converged Successfully: True
Iterations Required: 4
Average Out-of-Sample Log-Likelihood: -2.3011



Visual Evaluation and Mathematical Alignment

The background contour maps illustrate the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ derived in Part 3:

Regions of Certainty: Deep inside the cluster cores, the background color transitions to a dark, solid hue. In these regions, the responsibility for one component approaches $1.0$ (e.g., $\gamma_{i} \approx [0.98, 0.01, 0.01]$). This indicates high confidence, where the soft assignment vector aligns closely with a one-hot encoded vector.

Regions of Ambiguity: Along the decision boundaries where clusters meet, the contour values drop toward their theoretical minimum (for $K=3$, this boundary value approaches $1/3 \approx 0.33$). The lighter valleys between clusters visually highlight regions of high structural ambiguity. Here, the conditional expectation vector is spread evenly across components (e.g., $\gamma_{i} \approx [0.48, 0.52, 0.0]$), capturing the blend of cluster memberships before a hard decision is made.